In [1]:
!pip install chromadb sentence-transformers tavily-python

  Obtaining dependency information for tavily-python from https://files.pythonhosted.org/packages/63/ce/37e3aba0f359f540bfc57eb178f73d521161761f21e0aa28749f42750b11/tavily_python-0.7.24-py3-none-any.whl.metadata
  Obtaining dependency information for tiktoken>=0.5.1 from https://files.pythonhosted.org/packages/81/10/b8523105c590c5b8349f2587e2fdfe51a69544bd5a76295fc20f2374f470/tiktoken-0.12.0-cp312-cp312-win_amd64.whl.metadata
   ---------------------------------------- 0.0/878.7 kB ? eta -:--:--
   ------------------ --------------------- 399.4/878.7 kB 8.3 MB/s eta 0:00:01
   ---------------------------------------  870.4/878.7 kB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 878.7/878.7 kB 7.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import re
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import List, Optional

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
from tavily import TavilyClient

c:\Users\rachelle.l.macaraig\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(name="games_collection")

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5077.07it/s]


In [13]:
tavily = TavilyClient(api_key="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")

In [14]:
class State(Enum):
    START = auto()
    REWRITE = auto()        # use history to expand follow-ups
    RETRIEVE = auto()       # vector DB lookup
    EVALUATE = auto()       # score retrieval quality
    DECIDE = auto()         # branch on evaluation
    WEB_FALLBACK = auto()   # tavily search
    RESPOND = auto()        # finalize answer + commit to history
    DONE = auto()


@dataclass
class Turn:
    """One user/agent exchange we can refer back to later."""
    query: str
    rewritten_query: str
    answer: str
    source: str
    topic: Optional[str] = None  # e.g. "Elden Ring"


@dataclass
class TurnContext:
    """Mutable scratchpad for a single trip through the state machine."""
    raw_query: str
    rewritten_query: str = ""
    retrieved: List[str] = field(default_factory=list)
    distances: List[float] = field(default_factory=list)
    is_good: bool = False
    answer: str = ""
    source: str = ""
    topic: Optional[str] = None

In [15]:
SIM_THRESHOLD = 1.0

STOPWORDS = {"the", "a", "an", "is", "are", "was", "were", "of", "in", "on",
             "to", "for", "and", "or", "what", "who", "when", "where", "why",
             "how", "tell", "me", "about", "it", "its", "that", "this"}

# Words allowed inside a multi-word title (e.g. "God of War", "The Legend of Zelda").
CONNECTORS = {"of", "the", "and", "in", "on", "for", "to", "a", "an",
              "de", "la", "le", "du", "el"}

SENTENCE_STARTERS = {"Tell", "What", "Who", "When", "Where", "Why", "How",
                     "I", "Is", "Are", "Do", "Does", "Can", "Could", "Would"}

_REF_RE = re.compile(
    r"\b(it|its|that game|this game|the previous one|the same one)\b",
    flags=re.IGNORECASE,
)

In [16]:
def retrieve_games(query: str, n_results: int = 3):
    """Vector DB lookup -> (documents, distances)."""
    embedding = model.encode([query])[0]
    results = collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=n_results,
    )
    docs = results["documents"][0] if results.get("documents") else []
    dists = results["distances"][0] if results.get("distances") else []
    return docs, dists


def evaluate_results(query: str, docs: List[str], dists: List[float]) -> bool:
    """Accept DB results when EITHER similarity OR keyword overlap is strong."""
    if not docs:
        return False
    if dists and dists[0] < SIM_THRESHOLD:
        return True
    qwords = {w for w in query.lower().split() if w not in STOPWORDS}
    if not qwords:
        return False
    overlap = sum(len(qwords & set(d.lower().split())) for d in docs)
    return (overlap / len(docs)) >= 1


def web_search(query: str):
    response = tavily.search(query=query, max_results=3)
    return [{"content": r["content"], "url": r["url"]} for r in response["results"]]


def looks_like_followup(query: str) -> bool:
    """Short query that contains a pronoun or a back-reference."""
    q = query.lower().strip()
    if _REF_RE.search(q) and len(q.split()) <= 10:
        return True
    if len(q.split()) <= 4 and any(
        w in {"it", "its", "that", "this", "they", "them"} for w in q.split()
    ):
        return True
    return False


def extract_topic(text: str) -> Optional[str]:
    """Find capitalized phrases, allowing lowercase connectors like 'of'/'the'.

    'God of War Ragnarok' -> 'God of War Ragnarok' (not 'War Ragnarok').
    'Tell me about Elden Ring' -> 'Elden Ring' (sentence-starter dropped).
    """
    runs, current = [], []
    for word in text.split():
        if re.match(r"^[A-Z][a-zA-Z0-9']*$", word):
            current.append(word)
        elif word.lower() in CONNECTORS and current:
            current.append(word)
        else:
            while current and current[-1].lower() in CONNECTORS:
                current.pop()
            if len(current) >= 2:
                runs.append(" ".join(current))
            current = []
    while current and current[-1].lower() in CONNECTORS:
        current.pop()
    if len(current) >= 2:
        runs.append(" ".join(current))

    runs = [r for r in runs if r.split()[0] not in SENTENCE_STARTERS]
    if runs:
        return max(runs, key=len)

    for word in text.split():
        if re.match(r"^[A-Z][a-zA-Z0-9']*$", word) and word not in SENTENCE_STARTERS:
            return word
    return None

In [20]:
class GameAgent:
    def __init__(self) -> None:
        self.history: List[Turn] = []

    # ---- state handlers ----------------------------------------------------
    def _rewrite(self, ctx: TurnContext) -> State:
        if self.history and looks_like_followup(ctx.raw_query):
            last = self.history[-1]
            topic = last.topic or extract_topic(last.answer) or last.rewritten_query
            if topic:
                rewritten = _REF_RE.sub(topic, ctx.raw_query)
                if rewritten == ctx.raw_query:
                    rewritten = f"{ctx.raw_query.rstrip('?')} (about {topic})?"
                ctx.rewritten_query = rewritten
                print(f"🔄 Follow-up resolved: '{ctx.raw_query}' -> '{rewritten}'")
                return State.RETRIEVE

        ctx.rewritten_query = ctx.raw_query
        return State.RETRIEVE

    def _retrieve(self, ctx: TurnContext) -> State:
        print(f"🔧 Tool: Vector DB Retrieval  (query='{ctx.rewritten_query}')")
        ctx.retrieved, ctx.distances = retrieve_games(ctx.rewritten_query)
        if not ctx.retrieved:
            print("   -> Vector DB returned 0 documents (collection empty?)")
        else:
            for i, (doc, dist) in enumerate(zip(ctx.retrieved, ctx.distances), 1):
                snippet = doc.replace("\n", " ")[:90]
                ellipsis = "..." if len(doc) > 90 else ""
                print(f"   {i}. dist={dist:.3f}  {snippet}{ellipsis}")
        return State.EVALUATE

    def _evaluate(self, ctx: TurnContext) -> State:
        ctx.is_good = evaluate_results(ctx.rewritten_query, ctx.retrieved, ctx.distances)
        verdict = "Good — using vector DB" if ctx.is_good else "Poor — will fall back to web"
        print(f"🧪 Evaluation: {verdict}")
        return State.DECIDE

    def _decide(self, ctx: TurnContext) -> State:
        return State.RESPOND if ctx.is_good else State.WEB_FALLBACK

    def _web_fallback(self, ctx: TurnContext) -> State:
        print("🌐 Tool: Web Search (Tavily)")
        results = web_search(ctx.rewritten_query)
        if results:
            ctx.answer = results[0]["content"]
            ctx.source = results[0]["url"]
        else:
            ctx.answer = "No relevant results found."
            ctx.source = "n/a"
        return State.RESPOND

    def _respond(self, ctx: TurnContext) -> State:
        # When answering from the DB, keep only docs that meet the similarity
        # threshold. Chroma always returns n_results, even when most are noise.
        if ctx.is_good and not ctx.source:
            relevant = [
                d for d, dist in zip(ctx.retrieved, ctx.distances)
                if dist < SIM_THRESHOLD
            ]
            if not relevant:
                relevant = [ctx.retrieved[0]]
            ctx.answer = "\n\n---\n\n".join(relevant)
            ctx.source = "Internal Database"

        ctx.topic = (
            extract_topic(ctx.rewritten_query)
            or extract_topic(ctx.answer)
            or (self.history[-1].topic if self.history else None)
        )

        self.history.append(Turn(
            query=ctx.raw_query,
            rewritten_query=ctx.rewritten_query,
            answer=ctx.answer,
            source=ctx.source,
            topic=ctx.topic,
        ))
        return State.DONE

    # ---- driver loop -------------------------------------------------------
    def run(self, query: str) -> str:
        print(f"\n🧑 User: {query}")
        ctx = TurnContext(raw_query=query)

        handlers = {
            State.START: lambda c: State.REWRITE,
            State.REWRITE: self._rewrite,
            State.RETRIEVE: self._retrieve,
            State.EVALUATE: self._evaluate,
            State.DECIDE: self._decide,
            State.WEB_FALLBACK: self._web_fallback,
            State.RESPOND: self._respond,
        }

        state = State.START
        while state != State.DONE:
            state = handlers[state](ctx)

        last = self.history[-1]
        print("\n✅ Final Answer:")
        print(last.answer)
        print(f"\n📌 Source: {last.source}")
        if last.topic:
            print(f"🏷  Topic remembered for next turn: {last.topic}")
        return last.answer

In [24]:
if __name__ == "__main__":
    agent = GameAgent()

    # Turn 1 — establishes the topic (likely web fallback if Elden Ring isn't in DB)
    agent.run("Tell me about Elden Ring")

    # Turn 2 — pronoun "it" must resolve to "Elden Ring"
    agent.run("Who published it?")

    # Turn 3 — another pronoun follow-up; still about Elden Ring
    agent.run("When was it released?")

    # Turn 4 — vector DB hit; answer should contain ONLY God of War Ragnarok
    agent.run("What is the platform for God of War Ragnarok")

    # Turn 5 — pronoun follow-up on the DB-resolved topic
    agent.run("What genre is it?")

    print("\n--- Conversation history ---")
    for i, t in enumerate(agent.history, 1):
        print(f"{i}. raw='{t.query}'  rewritten='{t.rewritten_query}'  topic={t.topic}")

    print("\n--- Show that state was maintained ---")
    print(f"Total turns stored: {len(agent.history)}")
    for i, turns in enumerate(agent.history, 1):
        print(f"Turn {i}: {turns.query}")


🧑 User: Tell me about Elden Ring
🔧 Tool: Vector DB Retrieval  (query='Tell me about Elden Ring')
   1. dist=1.872  Title: God of War Ragnarok     Genre: Action     Platform: PS4, PS5     Description:
   2. dist=1.892  Title: Pokemon Red     Genre: RPG     Platform: Game Boy     Description:
   3. dist=2.015  Title: FIFA 21     Genre: Sports     Platform: PC, PS4, Xbox One     Description:
🧪 Evaluation: Poor — will fall back to web
🌐 Tool: Web Search (Tavily)

✅ Final Answer:
The **Elden Ring Wiki** covers Elden Ring, developed by FromSoftware and BANDAI NAMCO Entertainment Inc, is a fantasy action-RPG adventure set within a world created by Hidetaka Miyazaki -creator of the influential DARK SOULS video game series, and George R R Martin - author of The New York Times best-selling fantasy series, A Song of Ice and Fire. It is a game set within the universe or setting of Elden Ring that (might we say) features a "condensed" area of Limgrave where players will get to explore familiar are

## Agent Performance Report

Interaction Summary

 Total Turns: 5
 Topics Covered:
    Elden Ring (Turns 1–3)
    God of War Ragnarök (Turns 4–5)

Key Observations

✅ State Retention
The agent successfully maintained conversational context across three consecutive turns related to Elden Ring:

Initial inquiry about the game
Follow-up on publisher
Follow-up on release date

Each follow-up question was correctly interpreted as referring to the same subject without requiring restatement by the user.

✅ Context-Aware Responses
Questions such as “Who published it?” and “When was it released?” demonstrate correct anaphora resolution (“it”) based on prior turns.
This confirms effective short-term memory and proper use of conversational history.

✅ Topic Switching
At Turn 4, the user introduced a new, unrelated topic (God of War Ragnarök).
The agent correctly handled the topic transition without incorrectly carrying over context from the previous subject.
Follow-up question “What genre is it?” (Turn 5) was correctly scoped to the new topic, indicating successful context reset and re-anchoring.


Overall Assessment

✅ The agent demonstrates reliable state management, effective context tracking, and correct handling of follow-up queries and topic switches.
This performance indicates production-ready conversational behavior for multi-turn Q&A scenarios.